# Locate Test Blocks by Program (v1)

**Author:** Aaron Roodman  
**Date Created:** 2026-01-27  
**Last Modified:** 2026-03-14  
**Status:** In Progress  
**Keywords:** Program, Test Blocks, Observing Blocks, ConsDB  

## Description

Queries the ConsDB Visit1 table for all observations in a date range, maps BLOCK test case IDs
to human-readable names via the Zephyr Scale API, and tabulates band counts by day_obs for
selected test blocks.

Key functionality:
1. Query all Visit1 data from ConsDB for the specified date range
2. Look up Zephyr Scale test case names for each BLOCK science_program
3. Print band-count tables by day_obs for selected test blocks

**Output:** `test_block_names.txt` — mapping of BLOCK IDs to test case names  
**Based on:** Code from Lynne Jones for Zephyr Scale lookups

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-01-27 | Aaron Roodman | Initial version |
| 2026-03-14 | Aaron Roodman | Restructured to template; extended date range to 2026-03-14 |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Analysis](#analysis)
6. [Results & Plots](#results)

<a id='params'></a>
## Parameters

In [1]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

day_obs_min = 20250415
day_obs_max = 20260408

instrument = 'lsstcam'
bands = {'u': '#0c71ff', 'g': '#49be61', 'r': '#c61c00', 'i': '#ffc200', 'z': '#f341a2', 'y': '#5d0000'}

output_dir = 'output'


<a id='setup'></a>
## Setup & Imports

In [2]:
import os
import sys
import re
from pathlib import Path

os.environ["no_proxy"] += ",.consdb"

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import requests
from tqdm.notebook import tqdm

from astropy.time import Time, TimeDelta
from lsst.obs.lsst import LsstCam
from lsst.summit.utils import (
    ConsDbClient,
    getAirmassSeeingCorrection,
    getBandpassSeeingCorrection,
)
from lsst.summit.utils.efdUtils import (
    getEfdData,
    getMostRecentRowWithDataBefore,
    makeEfdClient,
)

# Add repo root to path for common imports
sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting

setup_plotting()

<a id='functions'></a>
## Helper Functions

In [3]:
def dayObsToMJD(day_obs):
    year = np.floor(day_obs / 10000).astype(int)
    month = np.floor((day_obs % 10000) / 100).astype(int)
    day = day_obs % 100
    time = Time({'year': year, 'month': month, 'day': day}, format='ymdhms')
    return time.mjd

def MJDToDayObs(mjd):
    time = Time(mjd, format='mjd')
    return np.array([_.iso.split()[0].replace('-', '') for _ in time])    

In [4]:
def print_band_counts_by_day(df, block_names, img_type_value):
    """
    Print a table showing band counts by day_obs for filtered data.
    
    Parameters
    ----------
    df : pandas.DataFrame
        The input DataFrame
    block_names : str or list of str
        Substring(s) to match in science_program column
    img_type_value : str
        Value to match in img_type column
    """
    # Convert single string to list for consistent handling
    if isinstance(block_names, str):
        block_names = [block_names]
    
    # Filter for specified blocks and img_type
    # Create mask for any of the block names
    block_mask = df['science_program'].str.contains(block_names[0], na=False)
    for block_name in block_names[1:]:
        block_mask |= df['science_program'].str.contains(block_name, na=False)
    
    filtered = df[block_mask & (df['img_type'] == img_type_value)].copy()
    
    print(f"Total rows matching {block_names} and img_type='{img_type_value}': {len(filtered)}")
    
    if len(filtered) == 0:
        print("No matching rows found.")
        return
    
    # Extract first character of band
    filtered['band_first'] = filtered['band'].str[0]
    
    # Create crosstab with day_obs as rows and first character of band as columns
    band_table = pd.crosstab(filtered['day_obs'], filtered['band_first'])
    
    # Reorder columns to u, g, r, i, z, y if they exist
    desired_order = ['u', 'g', 'r', 'i', 'z', 'y']
    band_table = band_table.reindex(columns=desired_order, fill_value=0)
    
    # Add totals row
    totals = band_table.sum(axis=0)
    band_table.loc['TOTAL'] = totals
    
    print("\nBand counts by day_obs:")
    print(band_table)

# Example usage:
# print_band_counts_by_day(df, 'BLOCK', 'SKYEXP')
# print_band_counts_by_day(df, ['BLOCK-123', 'BLOCK-456'], 'SKYEXP')

<a id='data'></a>
## Data Access

In [5]:
# Translate test block numbers above into more meaningful programs
jira_base_url = "https://rubinobs.atlassian.net/projects/BLOCK?selectedItem=com.atlassian.plugins.atlassian-connect-plugin:com.kanoah.test-manager__main-project-page#!/v2/testCase/"
with open("/home/r/roodman/.lsst/zephyr_token", "r") as f:
    zephyr_token = f.read().rstrip("\n")
zephyr_url = "https://api.zephyrscale.smartbear.com/v2/testcases/"
headers = {"Accept": "application/json",
           "Authorization": f"Bearer {zephyr_token}",
           "Content-Type": "application/json"} 

# note that this token is only good for 1 year.  go to https://rubinobs.atlassian.net/plugins/servlet/ac/com.kanoah.test-manager/api-access-tokens?project.key=BLOCK&project.id=10064
# to renew

In [6]:
client = ConsDbClient("http://consdb-pq.consdb:8080/consdb")

In [7]:
# get all Visit1 info in this day_obs range, joined with quicklook for PSF info
visits_query = f'''
    SELECT 
        v1.*,
        ql.physical_rotator_angle,
        ql.seeing_zenith_500nm_median,
        ql.psf_sigma_median,
        ql.n_psf_star_median,
        ql.psf_ixx_median,
        ql.psf_iyy_median,
        ql.psf_ixy_median
    FROM 
        cdb_{instrument}.visit1 v1
    LEFT JOIN
        cdb_{instrument}.visit1_quicklook ql
    ON v1.visit_id = ql.visit_id
    WHERE 
        v1.day_obs >= {day_obs_min} AND v1.day_obs <= {day_obs_max}
'''

visits = client.query(visits_query).to_pandas()


In [8]:
# some notes:
# scheduler_note is Observation Annotation in RubinTV

print(sorted(visits.columns))

['air_temp', 'airmass', 'altitude', 'altitude_end', 'altitude_start', 'azimuth', 'azimuth_end', 'azimuth_start', 'band', 'can_see_sky', 'controller', 'cur_index', 'dark_time', 'day_obs', 'dimm_seeing', 'emulated', 'exp_midpt', 'exp_midpt_mjd', 'exp_time', 'exposure_name', 'focus_z', 'group_id', 'humidity', 'img_type', 'max_index', 'n_psf_star_median', 'obs_end', 'obs_end_mjd', 'obs_start', 'obs_start_mjd', 'observation_reason', 'pgs_region', 'physical_filter', 'physical_rotator_angle', 'pressure', 'psf_ixx_median', 'psf_ixy_median', 'psf_iyy_median', 'psf_sigma_median', 's_dec', 's_ra', 's_region', 'scheduler_note', 'science_program', 'seeing_zenith_500nm_median', 'seq_num', 'shut_time', 'simulated', 'sky_rotation', 'target_name', 'vignette', 'vignette_min', 'visit_id', 'wind_dir', 'wind_speed', 'zenith_distance', 'zenith_distance_end', 'zenith_distance_start']


In [9]:
# code from Lynne Jones to get the Zephyr scale title for each science_program
if len(visits) > 0:
    test_names = {}
    jira_urls = {}
    block_programs = [sp for sp in visits.science_program.unique()
                      if sp is not None and sp.startswith("BLOCK-T")]
    for science_program in tqdm(block_programs, desc="Fetching Zephyr names"):
        # We have science_programs of the form BLOCK-T539_hexapods or _v3 .. anything post '_' is unusable
        response = requests.get(url=zephyr_url + science_program.split("_")[0], headers=headers)
        test_name = response.json()['name']
        jira_url = jira_base_url + science_program
        
        test_names[science_program] = test_name
        jira_urls[science_program] = jira_url


<a id='analysis'></a>
## Analysis

In [10]:
program_list = visits['science_program'].value_counts().sort_index()

In [11]:
# To display all entries temporarily for one Series
with pd.option_context('display.max_rows', None):
    print(program_list)

science_program
AOSSEQUENCE                        39
BLOCK-351                          71
BLOCK-365                       23147
BLOCK-368                           4
BLOCK-378                          16
BLOCK-387                          52
BLOCK-390                          16
BLOCK-395                           1
BLOCK-407                       10895
BLOCK-408                         610
BLOCK-416                          61
BLOCK-417                        3154
BLOCK-419                        6715
BLOCK-421                        6833
BLOCK-487                         106
BLOCK-518                         361
BLOCK-520                         130
BLOCK-544                           2
BLOCK-594                           1
BLOCK-640                           2
BLOCK-E8024                       401
BLOCK-R308                          4
BLOCK-T157                         38
BLOCK-T197                        273
BLOCK-T250                        598
BLOCK-T251                        

### Filter LUT BLOCK-T559

In [12]:
print_band_counts_by_day(visits, 'T559', 'acq')

Total rows matching ['T559'] and img_type='acq': 2477

Band counts by day_obs:
band_first   u    g     r    i    z    y
day_obs                                 
20250708     0   11    11   22   11   11
20250809     0    0    10   10   10   10
20250903     0    0    33    0    0    0
20250921     0   10    10   10   14    0
20251027     0   10    10   10   10    0
20251102     0   10    27   10   10   10
20251103     0   10    25   10   10   10
20251104     0   15    40   10   10   10
20251105     0    5    20    5    5    5
20251107     0   10    34   10   10   10
20251109     0   10    25   10   10   10
20251110     0   10    25   10   10   10
20251111     0    5    20    5    5    0
20251113     0   12    24   12    0    0
20251115    12    0    24    0   12    0
20251116     0   12    24   13    0    0
20251117    12    0    24    0   12    0
20251118     0    0    13    0   12    0
20251119     0    0     3    0    0    0
20251120     0    0     1    0    0    0
20251122     0   12

In [13]:
print_band_counts_by_day(visits, 'T559', 'cwfs')

Total rows matching ['T559'] and img_type='cwfs': 448

Band counts by day_obs:
band_first   u   g    r   i   z  y
day_obs                           
20251211     0   0   39  30   0  0
20251212     0   0   64  48   0  0
20251215    80   0  100   0   0  0
20251218     0  23   28   0   0  0
20251222     0   0   20   0  16  0
TOTAL       80  23  251  78  16  0


### BLOCK-T660 FAM Donuts during FBS

In [14]:
print_band_counts_by_day(visits, 'T630', 'cwfs')

Total rows matching ['T630'] and img_type='cwfs': 238

Band counts by day_obs:
band_first  u   g   r   i   z   y
day_obs                          
20251101    0   0   2   0   0   2
20251102    0   0   2   0   0  10
20251104    0   0   0   0   4   4
20251105    0   2   0   0   2   0
20251107    0   0   0   0   2   0
20251108    0   0   2   2   2   2
20251109    0   2   0   2   4   0
20251110    0   0   0   0   2   0
20251111    0   2   0   0   0   0
20251116    0   0   2   2   2   0
20251117    2   0   0   0   0   0
20251122    0   0   4   0   2   0
20251123    2   0   0   0   0   0
20251124    0   0   0   0   2   0
20251126    0   0   0   8   0   0
20251127    0   0   0  14   0   0
20251128    0   0   0  14   0   0
20251129    0   0   0   0  20   0
20251201    0   2   0   0   2   0
20251202    0   0   0   0   4   0
20251203    0   0   0   0   2   0
20251209    0   0   0   2   0   0
20251210    0   0   0   0   6   0
20251211    0   0   0   0   2   0
20251214    0   0   0  14   0   0
202

###  FAM Used in Analysis from Bryce

In [15]:
programs_touse = ['T278','T381','T492','T539','T614']
with pd.option_context('display.max_rows', None):
    print_band_counts_by_day(visits, programs_touse, 'cwfs')

Total rows matching ['T278', 'T381', 'T492', 'T539', 'T614'] and img_type='cwfs': 3411

Band counts by day_obs:
band_first  u    g     r     i   z   y
day_obs                               
20250505    0    0     2     0   0   0
20250512    0    0    28     0   0   0
20250519    0    0     0    10   0   0
20250521    0    0    38     0  13   0
20250522    0    0     0    15   0   0
20250524    0    0    62     0   0   0
20250525    0   28    31     0   0   0
20250527    0    0     0    28   0   0
20250529    0    0    50     0   0   0
20250531    0    0    24     0   0   0
20250601    0    0    32     0   0   0
20250706    0    0     0     0  12   0
20250707    0    0     0     0  24   0
20250825    0    0    81     0   0   0
20250826    0    0   100     0   0   0
20250907    0    0     0    12   0   0
20250909    0    0    14     0   0   0
20250912    0    0     0    20   0   0
20250913    0    0   200     0   0   0
20251023    0    0    48     0   0   0
20251024    0    0     0    11

###  Stability in-focus

In [16]:
programs_touse = ['T614']
with pd.option_context('display.max_rows', None):
    print_band_counts_by_day(visits, programs_touse, 'engtest')

Total rows matching ['T614'] and img_type='engtest': 1589

Band counts by day_obs:
band_first  u    g     r    i   z   y
day_obs                              
20250905    0    0    80    0   0   0
20250906    0    0    80    0   0   0
20250909    0    0    80    0   0   0
20250913    0    0    80    0   0   0
20250919    0    0     0    6   0   0
20251022    0    0     0   40   0   0
20251023    0    0    80    0   0   0
20251026    0   80     0   80   0   0
20251101    0    0    40    0   0  40
20251102    0    0    40    0   0   0
20251103    0    0    40    0   0   0
20251104    0    0    40    0   0   0
20251105    0    0    80    0   0   0
20251107    0    0     0   84   0   0
20251109    0    0    80    0   0   0
20251110    0    0    99    0   0   0
20251111    0    0    40    0  40   0
20251112    0    0     0   40   0   0
20251113    0   40     0    0   0   0
20251114    0    0     0   40   0   0
20251115    0    0     0    0  40   0
20251117    0    0    40    0   0   0
20251

<a id='results'></a>
## Results & Plots

In [17]:
# Export selected columns to gzipped CSV for offline analysis
# Filter out calibration frames (bias, dark, flat)
calib_types = ['bias', 'dark', 'flat']
visits_onsky = visits[~visits['img_type'].str.lower().isin(calib_types)].copy()

# Compute PSF ellipticity from second moments
ixx = pd.to_numeric(visits_onsky['psf_ixx_median'], errors='coerce')
iyy = pd.to_numeric(visits_onsky['psf_iyy_median'], errors='coerce')
ixy = pd.to_numeric(visits_onsky['psf_ixy_median'], errors='coerce')
visits_onsky['psf_ellipticity'] = np.sqrt((ixx - iyy)**2 + (2 * ixy)**2) / (ixx + iyy)

keep_cols = [
    'visit_id', 'day_obs', 'seq_num', 'band', 'obs_start', 'exp_time',
    's_ra', 's_dec', 'sky_rotation', 'physical_rotator_angle',
    'azimuth', 'altitude',
    'science_program', 'observation_reason', 'target_name', 'img_type',
    'seeing_zenith_500nm_median', 'psf_sigma_median', 'n_psf_star_median',
    'psf_ellipticity',
]
visits_export = visits_onsky[keep_cols].copy()

os.makedirs(output_dir, exist_ok=True)
csv_file = os.path.join(output_dir, 'visits_table.csv.gz')
visits_export.to_csv(csv_file, index=False, compression='gzip')
print(f'Wrote {len(visits_export)} rows x {len(keep_cols)} cols to {csv_file}')
print(f'(Filtered out {len(visits) - len(visits_onsky)} calibration frames)')


Wrote 102506 rows x 20 cols to output/visits_table.csv.gz
(Filtered out 51664 calibration frames)


In [ ]:
# Query butler for FAM danish triplet outputs
import subprocess

butler_cmd = [
    "butler", "query-datasets", "/repo/embargo",
    "aggregateAOSVisitTableRaw",
    "--collections", "aos_fam_danish_triplets",
    "--limit", "0",
]

result = subprocess.run(butler_cmd, capture_output=True, text=True)
butler_file = os.path.join(output_dir, "butler_fam_datasets.txt")
os.makedirs(output_dir, exist_ok=True)
with open(butler_file, "w") as f:
    f.write(result.stdout)

n_lines = result.stdout.count("\n")
print(f"Wrote {n_lines} lines to {butler_file}")
if result.stderr:
    print(f"stderr: {result.stderr[:200]}")


In [18]:
# Group by prefix and get one example value
prefix_dict = {}
for key, value in test_names.items():
    # Remove _vN or _N pattern from the end
    prefix = re.sub(r'_v?\d+$', '', key)
    if prefix not in prefix_dict:
        prefix_dict[prefix] = value

# Write sorted prefixes with values to output file and print
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, 'test_block_names.txt')
with open(output_file, 'w') as f:
    for prefix in sorted(prefix_dict.keys()):
        line = f"{prefix}: {prefix_dict[prefix]}"
        print(line)
        f.write(f"{line}\n")

print(f"\nWrote {len(prefix_dict)} unique prefixes to {output_file}")

BLOCK-T157: CLT-001 Closed Loop Converge - Kick z11
BLOCK-T197: ASCT-008 DOFs Degeneracy Test
BLOCK-T250: TMA Unpark/Checkout
BLOCK-T251: Pinhole Images for Scattered Light Characterization (ON-SKY)
BLOCK-T278: Wavefront Estimation Repeatability
BLOCK-T288: Acquire data for pointing model with Simonyi
BLOCK-T318: Bright star scans with ComCam.
BLOCK-T330: CBP Focus Test
BLOCK-T333: CBP Pointing Test
BLOCK-T339: CBP Filter Sweep Test Case
BLOCK-T339_red_leak: CBP Filter Sweep Test Case
BLOCK-T348: CBP Non-Linear Crosstalk
BLOCK-T373: Simonyi LSSTCam FBS Field Survey: Low Ecliptic Latitude Field at Opposition (Rubin_SV_216_-17)
BLOCK-T375: Camera Verification: Dark Exposure Testing Stage 1
BLOCK-T377: ASCT-004 Sensitivity Camera Hexapod
BLOCK-T378: ASCT-004 Sensitivity M1M3 Bending
BLOCK-T378_M1M3B12: ASCT-004 Sensitivity M1M3 Bending
BLOCK-T378_M1M3B20: ASCT-004 Sensitivity M1M3 Bending
BLOCK-T379: ASCT-004 Sensitivity M2 Bending
BLOCK-T379_M2B18: ASCT-004 Sensitivity M2 Bending
BLOCK-T